# 🃏 Flashcard Quiz Agent — Demo Notebook

End-to-end demonstration of the adaptive flashcard agent across **5 scenarios**.

| Cell | Scenario | What it proves |
|------|----------|----------------|
| 1 | Environment Setup | Agent initialises, tools loaded |
| 2 | Scenario A — Multi-Tool Chaining | 3 `add_card` calls from one prompt |
| 3 | Scenario B — Intentional Failure | `quiz_me` → wrong answer → `record_answer` |
| 4 | Scenario C — Adaptive Targeting | Weakest card served on re-quiz |
| 5 | Scenario D — Memory Summary | `get_stats` + conversation context |

> Every `[⚙️ Agent paused to use tool: ...]` line is **proof of a real tool call**, not a chatbot text response. A plain chatbot cannot produce these traces.

---
## Cell 1: Environment Setup & Agent Initialization

In [ ]:
import os, sys, json
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GROQ_MODEL   = os.getenv('GROQ_MODEL', 'openai/gpt-oss-120b')

if not GROQ_API_KEY:
    raise EnvironmentError('GROQ_API_KEY missing. Copy .env.example to .env and set it.')

print(f'API key  : {GROQ_API_KEY[:8]}...')
print(f'Model    : {GROQ_MODEL}')

# Reset DB to ensure a clean slate for the demo
from tools.flashcards import FLASHCARD_DB, _save_db
FLASHCARD_DB.clear()
FLASHCARD_DB.update({'next_id': 1, 'cards': {}})
_save_db(FLASHCARD_DB)
print('DB       : reset to empty')

from core.agent import GroqAgent
agent = GroqAgent(api_key=GROQ_API_KEY, model=GROQ_MODEL)
print('Agent    : GroqAgent initialized — ReAct loop ready')

API key  : gsk_XXXX...
Model    : openai/gpt-oss-120b
DB       : reset to empty
Agent    : GroqAgent initialized — ReAct loop ready


---
## Cell 2: Scenario A — Data Ingestion (Multi-Tool Chaining)

**One prompt → three sequential `add_card` tool calls.**

The agent parses a single user message, realises it must perform three distinct actions, and fires `add_card` three separate times before generating its final confirmation reply. This proves the plan-act loop — not a one-shot chatbot.

> **Trace proof:** Look for three stacked `[⚙️ Agent paused to use tool: add_card]` lines.

In [ ]:
response_a = agent.chat(
    'Please add these three study cards for me:\n'
    '1. Q: What is the time complexity of searching in a balanced BST? '
       '/ A: O(log n)\n'
    '2. Q: What does ACID stand for in DBMS? '
       '/ A: Atomicity, Consistency, Isolation, Durability\n'
    '3. Q: What is the purpose of Docker? '
       '/ A: OS-level virtualization and containerization'
)
print('--- Agent Final Response ---')
print(response_a)


[⚙️  Agent paused to use tool: add_card]
[📦 Tool result]: {"status": "success", "message": "Flashcard #1 added successfully.", "card": {"id": 1, "question": "What is the time complexity of searching in a balanced BST?", "answer": "O(log n)"}}


[⚙️  Agent paused to use tool: add_card]
[📦 Tool result]: {"status": "success", "message": "Flashcard #2 added successfully.", "card": {"id": 2, "question": "What does ACID stand for in DBMS?", "answer": "Atomicity, Consistency, Isolation, Durability"}}


[⚙️  Agent paused to use tool: add_card]
[📦 Tool result]: {"status": "success", "message": "Flashcard #3 added successfully.", "card": {"id": 3, "question": "What is the purpose of Docker?", "answer": "OS-level virtualization and containerization"}}

--- Agent Final Response ---
All three flashcards have been saved! Here's your deck:

1. 🃏 **#1** — What is the time complexity of searching in a balanced BST?
2. 🃏 **#2** — What does ACID stand for in DBMS?
3. 🃏 **#3** — What is the purpose of Do

In [ ]:
# Verify all 3 cards are in the database
from tools.flashcards import FLASHCARD_DB
print(f'Cards in DB: {len(FLASHCARD_DB["cards"])}')
for cid, c in FLASHCARD_DB['cards'].items():
    print(f'  [{cid}] Q: {c["question"]!r}')
    print(f'       A: {c["answer"]!r} | errors={c["incorrect_count"]}')

Cards in DB: 3
  [1] Q: 'What is the time complexity of searching in a balanced BST?'
       A: 'O(log n)' | errors=0
  [2] Q: 'What does ACID stand for in DBMS?'
       A: 'Atomicity, Consistency, Isolation, Durability' | errors=0
  [3] Q: 'What is the purpose of Docker?'
       A: 'OS-level virtualization and containerization' | errors=0


---
## Cell 3: Scenario B — State Evaluation (Intentional Failure)

**Quiz trigger → wrong answer → `record_answer` increments `incorrect_count`.**

The agent calls `quiz_me`, fetches a card from the Python database, and presents only the question. We deliberately answer incorrectly. The agent evaluates the response against the stored answer field (not its own knowledge), calls `record_answer(is_correct=False)`, and increments that card's error count in the persistent state. This proves tool results drive the next action.

In [ ]:
# Step B1: Ask for a quiz question
response_b1 = agent.chat('Quiz me on one of my cards.')
print('--- Agent Response (question) ---')
print(response_b1)


[⚙️  Agent paused to use tool: quiz_me]
[📦 Tool result]: {"status": "success", "priority": "least_seen", "priority_reason": "No errors recorded yet. Serving the least-seen card (0 attempts).", "card": {"id": 1, "question": "What is the time complexity of searching in a balanced BST?", "answer": "O(log n)", "incorrect_count": 0, "total_attempts": 0}}

--- Agent Response (question) ---
📖 **Question:** What is the time complexity of searching in a balanced BST?

Take your time! 🤔


In [ ]:
# Step B2: Give a deliberately WRONG answer
response_b2 = agent.chat('ACID stands for Apple, Cat, Ice, and Dog.')
print('--- Agent Response (evaluation) ---')
print(response_b2)


[⚙️  Agent paused to use tool: record_answer]
[📦 Tool result]: {"status": "success", "card_id": 1, "is_correct": false, "updated_metrics": {"incorrect_count": 1, "total_attempts": 1}, "message": "\u274c Incorrect. Card #1 is now prioritised for review (total errors: 1)."}

--- Agent Response (evaluation) ---
❌ Not quite! The correct answer is: **O(log n)**

Card #1 has been flagged as a weak spot — I'll serve it again soon. Keep going! 💪


In [ ]:
# Confirm incorrect_count incremented in the database
from tools.flashcards import FLASHCARD_DB
c = FLASHCARD_DB['cards']['1']
print(f'Card #1 after wrong answer:')
print(f'  incorrect_count : {c["incorrect_count"]}  ← was 0, now 1')
print(f'  total_attempts  : {c["total_attempts"]}')

Card #1 after wrong answer:
  incorrect_count : 1  ← was 0, now 1
  total_attempts  : 1


---
## Cell 4: Scenario C — Closed-Loop Reasoning (Adaptive Targeting)

**This is the core proof of agentic behaviour.**

When asked to quiz again, the agent calls `quiz_me()`. The Python tool logic inspects all cards' `incorrect_count` values, finds Card #1 has the highest error count (1 vs 0 for all others), and returns it with `priority: "weak_spot"`. The agent is forced to re-serve your weakest card — not a random one.

> This cannot be faked by a chatbot. The adaptive selection happens inside the Python tool, driven by the persisted state — the model simply reads the result.

In [ ]:
# Step C1: Ask to be quizzed again — agent MUST target the weak card
response_c1 = agent.chat('Quiz me again.')
print('--- Agent Response (adaptive re-quiz) ---')
print(response_c1)


[⚙️  Agent paused to use tool: quiz_me]
[📦 Tool result]: {"status": "success", "priority": "weak_spot", "priority_reason": "This card has been answered incorrectly 1 time(s) — highest error count.", "card": {"id": 1, "question": "What is the time complexity of searching in a balanced BST?", "answer": "O(log n)", "incorrect_count": 1, "total_attempts": 1}}

--- Agent Response (adaptive re-quiz) ---
Serving your weakest card — you've missed it 1 time(s)! Let's fix that 💪

📖 **Question:** What is the time complexity of searching in a balanced BST?


### ✅ Adaptive Selection Confirmed

The tool result shows `"priority": "weak_spot"` — Card #1 was chosen **because** its `incorrect_count` is higher than all other cards, not at random.

Now let's answer correctly to prove the agent updates the metrics:

In [ ]:
# Step C2: Give the CORRECT answer
response_c2 = agent.chat('O(log n)')
print('--- Agent Response (correct answer) ---')
print(response_c2)


[⚙️  Agent paused to use tool: record_answer]
[📦 Tool result]: {"status": "success", "card_id": 1, "is_correct": true, "updated_metrics": {"incorrect_count": 1, "total_attempts": 2}, "message": "\u2705 Correct! Card metrics updated."}

--- Agent Response (correct answer) ---
✅ **Correct!** O(log n) is right — binary search on a balanced BST halves the search space at each step. Great job fixing your mistake! 🎉

Would you like another question?


---
## Cell 5: Scenario D — Contextual Memory Summary

**Sliding-window conversation context + `get_stats()` tool = accurate diagnostics.**

When the student asks which topic they are struggling with, the agent does two things:
1. Calls `get_stats()` to read the live database state (persistent memory)
2. Uses the conversation sliding window to recall earlier context (short-term memory)

The combination produces a reliable session summary — not a hallucinated one.

In [ ]:
response_d = agent.chat(
    'Give me a quick summary: how many cards do I have, '
    'and which topic did I struggle with most?'
)
print('--- Agent Response (summary) ---')
print(response_d)


[⚙️  Agent paused to use tool: get_stats]
[📦 Tool result]: {"status": "success", "total_cards": 3, "weakest_card": {"id": 1, "question": "What is the time complexity of searching in a balanced BST?", "incorrect_count": 1}, "all_cards": [{"id": 1, "question": "What is the time complexity of searching in a balanced BST?", "incorrect_count": 1, "total_attempts": 2, "accuracy": "50%"}, {"id": 2, "question": "What does ACID stand for in DBMS?", "incorrect_count": 0, "total_attempts": 0, "accuracy": "not attempted"}, {"id": 3, "question": "What is the purpose of Docker?", "incorrect_count": 0, "total_attempts": 0, "accuracy": "not attempted"}]}

--- Agent Response (summary) ---
Here's your session snapshot 📊

**Total cards:** 3

| # | Question | Accuracy | Errors |
|---|----------|----------|--------|
| 1 | What is the time complexity of searching in a balanced BST? | 50% | 1 |
| 2 | What does ACID stand for in DBMS? | not attempted | 0 |
| 3 | What is the purpose of Docker? | not attempted

---
## Summary

| Scenario | Tool calls | What was proved |
|----------|------------|----------------|
| A — Multi-tool chaining | `add_card` × 3 | One prompt → multiple autonomous tool calls |
| B — Intentional failure | `quiz_me` + `record_answer(False)` | Tool result drives next action; error count persisted |
| C — Adaptive targeting | `quiz_me` → `priority: weak_spot` | Agent reads state, serves weakest card not random |
| D — Memory summary | `get_stats` + conversation window | Dual memory: persistent DB + sliding-window context |

> **The notebook is the proof.** A plain chatbot generates text. This agent calls tools, reads the results, and loops autonomously — every `[⚙️ Agent paused to use tool: ...]` line is evidence of that.